In [ ]:
import urllib.request
import os
from datetime import datetime, date, timezone
from dateutil.relativedelta import relativedelta

from nyctaxi_project.transformations.modules.data_loader.file_downloader import download_file

# Obtains year-month from todays date to two months ago in format yyyy-mm

four_months_ago = date.today() - relativedelta(months=4)
formatted_date = four_months_ago.strftime("%Y-%m")

# Location of data for formatted_date
if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    data_location = f"/Volumes/nyctaxi/00_landing/data_sources/nyctaxi_yellow/{formatted_date}"
else:
    data_location = f"{os.getenv('LOCAL_UPLOADED_DATA_PATH')}/nyctaxi_yellow/{formatted_date}"

#full path to the parquet file
data_file = f"{data_location}/yellow_tripdata_{formatted_date}.parquet"

try:
    #check if file already exists
    dbutils.fs.ls(data_file)

    # if file exists set continue_downstream=no
    dbutils.jobs.taskValues.set(key="continue_downstream", value="no")
    print("File already exists")
except:
    try:
        #form a url to the file for data_file download
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{formatted_date}.parquet"

        # open the connection to the remote file and download
        response = urllib.request.urlopen(url)

        # create folder for this new file
        os.makedirs(data_location, exist_ok=True)

        #copy data from response to file
        download_file(url, data_file)
        
        # set continue_downstream to yes if file was loaded
        dbutils.jobs.taskValues.set(key="continue_downstream", value="yes")
        print("File downloaded")
    except Exception as e:
        dbutils.jobs.taskValues.set(key="continue_downstream", value="no")
        print(f"File download failed {str(e)}")
    
#



File downloaded
